# Week 9, Exercise 9: Coding a Sequence-to-Sequence Transformer

We implement an encoder-decoder Transformer, whose decoder attends to the encoder outputs through cross-attention, and train it to reverse a sequence of 8 digits.

This notebook contains the full solution of Exercise 9 from the [Week 9 optional exercises](https://utm-csc413.github.io/2026F-website/optional_exercises/w09/exercises_w09.html). The saved outputs come from a run on a CPU; your numbers may differ slightly on Colab or with other PyTorch versions. Each training run takes a few seconds to about half a minute on a CPU.

## Building blocks from Exercise 7

`MultiHeadAttention`, `causal_mask`, and `Block`, copied from Exercise 7.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):                       # (B, n, d_model) -> (B, h, n, d_k)
        B, n, _ = x.shape
        return x.view(B, n, self.h, self.d_k).transpose(1, 2)

    def forward(self, x_q, x_kv, mask=None):
        Q = self.split_heads(self.W_q(x_q))         # (B, h, n_q, d_k)
        K = self.split_heads(self.W_k(x_kv))        # (B, h, n_k, d_k)
        V = self.split_heads(self.W_v(x_kv))        # (B, h, n_k, d_k)
        scores = Q @ K.transpose(-2, -1) / self.d_k ** 0.5   # (B, h, n_q, n_k)
        if mask is not None:
            scores = scores + mask                  # mask: (n_q, n_k), 0 or -inf
        C = torch.softmax(scores, dim=-1) @ V       # (B, h, n_q, d_k)
        B, _, n_q, _ = C.shape
        C = C.transpose(1, 2).reshape(B, n_q, self.h * self.d_k)  # concatenate heads
        return self.W_o(C)

def causal_mask(n):
    return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)

class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        a = self.norm1(x + self.attn(x, x, mask))   # self-attention, Add & Norm
        return self.norm2(a + self.ffn(a))          # FFN, Add & Norm

## Data

The decoder input is the target shifted right, starting with `<BOS>`, and the decoder target ends with `<EOS>`. All sequences have the same length, so we need no padding mask.

In [2]:
BOS, EOS = 10, 11                                    # digits are tokens 0-9
vocab_size, n = 12, 8

def make_batch(batch_size):
    src = torch.randint(10, (batch_size, n))
    tgt = src.flip(dims=[1])                         # the reversed sequence
    bos = torch.full((batch_size, 1), BOS)
    eos = torch.full((batch_size, 1), EOS)
    tgt_in = torch.cat([bos, tgt], dim=1)            # <BOS> y1 ... yn   (shifted right)
    tgt_out = torch.cat([tgt, eos], dim=1)           # y1 ... yn <EOS>
    return src, tgt_in, tgt_out

## (a)-(b) The decoder block and the encoder-decoder model

In cross-attention, the queries come from the decoder and the keys and values come from the encoder outputs `z`.

In [3]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, s, z, mask):
        a = self.norm1(s + self.self_attn(s, s, mask))   # masked self-attention
        b = self.norm2(a + self.cross_attn(a, z))        # queries from a, keys/values from z
        return self.norm3(b + self.ffn(b))

class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, max_len, d_model=64, n_heads=4, d_ff=128, n_layers=2):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.encoder = nn.ModuleList([Block(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.out = nn.Linear(d_model, vocab_size)

    def embed(self, idx):
        return self.tok_emb(idx) + self.pos_emb(torch.arange(idx.shape[1]))

    def encode(self, src):
        z = self.embed(src)
        for block in self.encoder:
            z = block(z)                                 # no mask in the encoder
        return z

    def decode(self, tgt_in, z):
        s = self.embed(tgt_in)
        mask = causal_mask(tgt_in.shape[1])
        for block in self.decoder:
            s = block(s, z, mask)
        return self.out(s)                               # (B, m, vocab_size)

    def forward(self, src, tgt_in):
        return self.decode(tgt_in, self.encode(src))

## (c) Training with teacher forcing, and greedy decoding

In [4]:
torch.manual_seed(0)
model = Seq2SeqTransformer(vocab_size, max_len=n + 1)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for step in range(1000):
    src, tgt_in, tgt_out = make_batch(64)
    logits = model(src, tgt_in)                          # teacher forcing
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 250 == 0 or step == 999:
        print(f"step {step:4d}  loss {loss.item():.4f}")

@torch.no_grad()
def greedy_decode(model, src, n_new=n + 1):
    z = model.encode(src)                                # encode the source once
    out = torch.full((src.shape[0], 1), BOS)
    for _ in range(n_new):
        next_tok = model.decode(out, z)[:, -1].argmax(dim=-1, keepdim=True)
        out = torch.cat([out, next_tok], dim=1)
    return out[:, 1:]                                    # drop <BOS>

src, _, tgt_out = make_batch(1000)
pred = greedy_decode(model, src)
print("exact match:", (pred == tgt_out).all(dim=1).float().mean().item())
print("input: ", src[0].tolist())
print("output:", pred[0].tolist())

step    0  loss 2.7431


step  250  loss 0.0038


step  500  loss 0.0013


step  750  loss 0.0007


step  999  loss 0.0004
exact match: 1.0
input:  [2, 9, 6, 8, 3, 5, 3, 8]
output: [8, 3, 5, 3, 8, 6, 9, 2, 11]


The model reverses essentially every test sequence and ends each one with `<EOS>` (token 11).

## (d)

The encoder reads an input sequence that is entirely known in advance, so every input token can use context from both sides. The decoder is trained with teacher forcing to predict token $i+1$ from tokens $1, \dots, i$. Without the mask, it could read the answer from position $i+1$ of its input (see Exercise 7(e)), and at generation time the future tokens don't exist yet. In cross-attention, the queries come from the decoder (the output of its masked self-attention sublayer), and the keys and values come from the final encoder outputs $Z$. This is how each output position looks up the relevant input tokens; for reversal, output position $i$ learns to attend to input position $n + 1 - i$.